In [0]:
spclientid = dbutils.secrets.get(scope = "adls-scope", key = "sp-client-id")
spclientsecret = dbutils.secrets.get(scope = "adls-scope", key = "sp-client-secret")
spoid = dbutils.secrets.get(scope = "adls-scope", key = "sp-tenant-id")

In [0]:
# Get silver container path
silver_path = "abfss://silver@gurboapistorage.dfs.core.windows.net/CGY-Games"
# Read silver table
df = spark.read.format("delta").load(silver_path)
df.show()

In [0]:
from pyspark.sql.functions import col, when, sum

# total wins/losses
df_gold_winloss = df.agg(sum(when(col("CGYWin") == True, 1).otherwise(0)).alias("wins"), sum(when(col("CGYWin") == False, 1).otherwise(0)).alias("Losses"))

# positive diff = CGY dominated opponent, negative diff = opponent dominated CGY
df_goal_diff = df.groupby("OpponentTeamName").agg(sum(col("CGYScore") - col("OpponentScore")).alias("OpponentGoalDiff"))

df_gold_winloss.show()
df_goal_diff.show()

In [0]:
gold_WL_path = "abfss://gold@gurboapistorage.dfs.core.windows.net/CGY_WinLoss"
gold_dif_path = "abfss://gold@gurboapistorage.dfs.core.windows.net/CGY_GoalDiff"

# Write table to Gold container as delta format
df_gold_winloss.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_WL_path)
df_goal_diff.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold_dif_path)
